# 03 — Sistema A original, pesos fixos

Objetivo: executar o fluxo original e distinguir sem regiões, negativos classificados, positivos e erros. Entradas: manifesto; para inferência real, `TCC_MODEL_A` aponta para o artefato .keras. Saídas: tabela/JSON e sobreposições via CLI. Pré-requisitos: TensorFlow/Keras 2.15, Python 3.10 ou 3.11. O modo sintético usa dublê de classificador, não os pesos originais.

Execução: na raiz, `python -m jupyter lab notebooks`. Execute de cima para baixo. Variáveis: `TCC_ROOT`, `TCC_RUN_MODE=synthetic|real`, `TCC_MANIFEST`, `TCC_ANNOTATIONS`, `TCC_SPLIT`. Colab pode definir `TCC_ROOT` para a cópia do repositório e instalar as dependências em runtime Python compatível.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("TCC_ROOT", Path.cwd())).resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Defina TCC_ROOT para a raiz do repositório."
sys.path.insert(0, str(ROOT))
MODE = os.environ.get("TCC_RUN_MODE", "synthetic")
assert MODE in {"synthetic", "real"}
from scripts.notebook_support import prepare_synthetic, SyntheticClassifier
from src import data_protocol as dp
if MODE == "synthetic":
    OUTPUT, rows = prepare_synthetic(ROOT)
    MANIFEST = OUTPUT / "images.csv"
    ANNOTATIONS = OUTPUT / "synthetic_annotations.json"
else:
    OUTPUT = ROOT / "experiments" / "runs" / "notebook_real"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    MANIFEST = Path(os.environ.get("TCC_MANIFEST", ROOT / "data/splits/luiz/images.csv"))
    ANNOTATIONS = Path(os.environ.get("TCC_ANNOTATIONS", ROOT / "data/annotations/luiz_pending.json"))
print({"modo": MODE, "raiz": str(ROOT), "manifesto": str(MANIFEST)})


In [ ]:
from src.model_a import infer_image
if MODE == "synthetic":
    results = [infer_image(dp.read_image_rgb(row["path"])[:,:,::-1].copy(), SyntheticClassifier(), row["image_id"]) for row in rows]
    assert all(r["status"] != "erro" for r in results)
    print([(r["status"],len(r["regions"])) for r in results])
else:
    model = os.environ.get("TCC_MODEL_A")
    assert model, "Defina TCC_MODEL_A; pesos não são baixados nem treinados."
    subprocess.run([sys.executable, str(ROOT / "scripts/run_model_a.py"),
                    "--manifest", str(MANIFEST), "--model", model,
                    "--split", os.environ.get("TCC_SPLIT", "development"),
                    "--output-dir", str(OUTPUT / "A"), "--overlays"], check=True, cwd=ROOT)


Referência local congelada: VisionCruzi `59b10244f2ce5cfabff7bece404cb5679668a45d`. O mapeamento 0/1 foi confirmado no encoder e na API. Veja `docs/execucao_2026-09-20.md` para a nova execução real, hashes e limitações. Não chamar ausência de recortes de classificação negativa.